In [1]:
# 평가함수(classification_report)
# Accuracy, Recall, Precision, F1 score값을 계산

from sklearn.metrics import classification_report

# 정답 데이터와
# 예측 데이터가 있어야 해요!
t_data = [0, 1, 2, 2, 2]  # 0:thin, 1:normal, 2:fat
predict = [0, 0, 2, 2, 1] # 우리 모델의 예측값
target_name = ['thin', 'normal', 'fat']
print(classification_report(t_data,
                            predict,
                            target_names=target_name))

              precision    recall  f1-score   support

        thin       0.50      1.00      0.67         1
      normal       0.00      0.00      0.00         1
         fat       1.00      0.67      0.80         3

    accuracy                           0.60         5
   macro avg       0.50      0.56      0.49         5
weighted avg       0.70      0.60      0.61         5



In [10]:
# 대표적인 다중분류 문제인
# Iris 분류를 구현해 보아요!
# 필요한 모듈 import

import numpy as np
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris()
# print(iris.DESCR)
# print(iris.data)
df = pd.DataFrame(iris.data,
                  columns=iris.feature_names)
df.columns = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
df['label'] = iris.target
display(df)

,sepal_length,sepal_width,petal_length,petal_width,label
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...
145,6.7,3.0,5.2,2.3,2
146,6.3,2.5,5.0,1.9,2
147,6.5,3.0,5.2,2.0,2
148,6.2,3.4,5.4,2.3,2


In [17]:
# 데이터 전처리 및 확인
# df.info()
# df.isnull().sum() # 결측치 확인 - 결측치가 없어요!
# 중복데이터가 있는지 확인!
# print(df.duplicated().sum()) # 중복행이 없으면 당연히 0이 나와야 해요!
# 결과가 1이예요! 쓸데없이 중복된 데이터가 1개 존재해요!
df_new = df.drop_duplicates()
# df_new.shape # (149, 5)

(149, 5)

In [18]:
# 4개의 feature가 존재해요!
# 이 4개의 feature가 어느정도는 당연히 종속변수에 영향을 미쳐야 해요!
# 한번 독립변수들이 종속변수에 어느정도 연관성이 있는지를
# 수치로 확인해보아요!
# 상관관계를 분석하면 되요!
print(df.corr())
# 계산된 결과는 우리가 상관계수라고 말하는데
# 이 값은 -1 ~ 1사이의 실수로 계산되요!
# 1은 양의 상관관계, -1은 음의 상관관계
# 양 극단으로 값이 갈수록 상관관계가 더 많아요!

              sepal_length  sepal_width  petal_length  petal_width     label
sepal_length      1.000000    -0.117570      0.871754     0.817941  0.782561
sepal_width      -0.117570     1.000000     -0.428440    -0.366126 -0.426658
petal_length      0.871754    -0.428440      1.000000     0.962865  0.949035
petal_width       0.817941    -0.366126      0.962865     1.000000  0.956547
label             0.782561    -0.426658      0.949035     0.956547  1.000000


In [23]:
# Raw Data가 준비되었으니
# 데이터 전처리를 진행!

# 필요한 모듈 import
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy import stats
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.model_selection import train_test_split

from sklearn import linear_model
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Flatten, Dense
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import classification_report

In [ ]:
# df_new # 149 rows × 5 columns

# 1. 결측치는 없어요!
# 2. 이상치
# boxplot을 이용해서 일단 눈으로 확인해 보아요!
# fig = plt.figure()
# ax1 = fig.add_subplot(1,4,1)
# ax2 = fig.add_subplot(1,4,2)
# ax3 = fig.add_subplot(1,4,3)
# ax4 = fig.add_subplot(1,4,4)
# ax1.boxplot(df_new['sepal_length'])
# ax2.boxplot(df_new['sepal_width'])
# ax3.boxplot(df_new['petal_length'])
# ax4.boxplot(df_new['petal_width'])
# plt.tight_layout()  # 그래프를 조금 예쁘게 그려요!
# plt.show()
# boxplot을 확인하고 이상치 처리를 결정해야 해요!
# 지금은 이상치 처리를 제외하고 갈께요!
# 3. 정규화
# 거의 예외없이 진행해야 해요!
# MinMaxScaler, StandardScaler 이용
x_data = df_new.drop('label', axis=1, inplace=False).values
t_data = df_new['label'].values  # 1차원

# scaler = StandardScaler()
scaler = MinMaxScaler()
scaler.fit(x_data)
x_data_norm = scaler.transform(x_data)
print(x_data_norm)

# 4. 데이터 분할(평가를 진행해야 해요!)
# 약간 유연성을 좀 가지면서 처리해주세요!(데이터 부족때문에!!)
x_data_train_norm, x_data_test_norm, t_data_train, t_data_test = \
train_test_split(x_data_norm,
                 t_data,
                 test_size=0.2,
                 stratify=t_data)


In [29]:
# sklearn 다중 분류 모델 구현
sklearn_model = linear_model.LogisticRegression()
sklearn_model.fit(x_data_train_norm,
                  t_data_train) # one-hot 처리하지 않고 1차원으로 입력
# 모델 평가
sklearn_result = sklearn_model.predict(x_data_test_norm)
# print(sklearn_result)
print(classification_report(t_data_test,
                            sklearn_result,
                            target_names=['Setosa', 'Vesicolour', 'Virsinica']))

              precision    recall  f1-score   support

      Setosa       1.00      1.00      1.00        10
  Vesicolour       0.89      0.80      0.84        10
   Virsinica       0.82      0.90      0.86        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30



In [ ]:
# Tensorflow 구현
keras_model = Sequential()

keras_model.add(Flatten(input_shape=(4,)))
keras_model.add(Dense(units=3,
                      activation='softmax'))

# keras_model.summary()
keras_model.compile(optimizer=Adam(learning_rate=1e-1),
                    loss='sparse_categorical_crossentropy',
                    metrics=['acc'])

es_callback = EarlyStopping(monitor='val_loss',
                            patience=5,
                            restore_best_weights=True,
                            verbose=1)

history = keras_model.fit(x_data_train_norm,
                          t_data_train.reshape(-1,1),
                          epochs=1000,
                          verbose=1,
                          validation_split=0.2,
                          callbacks=[es_callback])

In [41]:
# 모델 평가
keras_result = keras_model.predict(x_data_test_norm)
# 2차원 확률값을 1차원 label값으로 변경
print(classification_report(t_data_test,
                            np.argmax(keras_result, axis=1),
                            target_names=['Setosa', 'Vesicolour', 'Virsinica']))


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
              precision    recall  f1-score   support

      Setosa       1.00      1.00      1.00        10
  Vesicolour       0.91      1.00      0.95        10
   Virsinica       1.00      0.90      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

